In [ ]:
import sqlite3
from pathlib import Path
import sys
import pandas as pd

sys.path.append('..')

DB_PATH = Path('..') / 'database' / 'codepulse.db'
conn = sqlite3.connect(DB_PATH)

events = pd.read_sql('SELECT * FROM events', conn)
pull_requests = pd.read_sql('SELECT * FROM pull_requests', conn)
pr_commits = pd.read_sql('SELECT * FROM pr_commits', conn)
issues = pd.read_sql('SELECT * FROM issues', conn)
commit_issues = pd.read_sql('SELECT * FROM commit_issues', conn)
try:
    commit_labels = pd.read_sql('SELECT * FROM commit_labels', conn)
except Exception:
    commit_labels = None

conn.close()

In [ ]:
events

In [ ]:
pull_requests

In [ ]:
pr_commits

In [ ]:
issues

In [ ]:
commit_issues

In [ ]:
commit_labels

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(commit_labels['bug_confidence'], bins=30, kde=True, ax=axes[0])
axes[0].set_title('Bug confidence — all commits')
axes[0].set_xlabel('bug_confidence')

nonzero = commit_labels[commit_labels['bug_confidence'] > 0]['bug_confidence']
sns.histplot(nonzero, bins=30, kde=True, color='orange', ax=axes[1])
axes[1].set_title(f'Bug confidence — non-zero only ({len(nonzero)} commits)')
axes[1].set_xlabel('bug_confidence')

plt.tight_layout()
plt.show()

## Sanity checks
How much of the data actually carries signal — and how noisy is our weakest signal (keyword-only)?

In [ ]:
# 1. What fraction of PRs have ANY label at all?
labeled = pull_requests['labels'].fillna('').str.len() > 0
print(f'PRs with at least one label: {labeled.sum()}/{len(pull_requests)} ({labeled.mean():.1%})')

from src.labeling.labeler import has_bug_label
bug_labeled = pull_requests['labels'].apply(has_bug_label)
print(f'PRs with a bug-pattern label:  {bug_labeled.sum()}/{len(pull_requests)} ({bug_labeled.mean():.1%})')

In [ ]:
# 2. Sample 15 keyword-only commits — are these real bugs or false positives?
keyword_only = commit_labels[commit_labels['signals_matched'] == 'keyword']
sample = keyword_only.merge(
    events[['commit_hash', 'commit_message']].drop_duplicates(),
    on='commit_hash'
)[['commit_hash', 'commit_message']].sample(15, random_state=42)

for _, row in sample.iterrows():
    short_hash = row['commit_hash'][:8]
    first_line = row['commit_message'].split('\n')[0][:90]
    print(f'  {short_hash}  {first_line}')

In [ ]:
# 3. What are the most common labels in this repo? Spot bug labels we might be missing.
all_labels = (pull_requests['labels'].fillna('')
              .str.split(',').explode().str.strip())
all_labels = all_labels[all_labels != '']
all_labels.value_counts().head(20)